# Setup VM

## Using Existing Instance

### Choose Site and Project 

In [ ]:
from chi import context

context.version = "1.0"

context.use_site("CHI@UC") #change to your site 
context.use_project("CHI-251412") # change to your porject

### Connect to Server

In [ ]:
from chi import server

my_server = server.get_server("maicbf-instance") #change to your instance name
print("Server Status : ", my_server.status)
my_server.check_connectivity()

In [ ]:
my_server.execute("python3 --version")

## Using New Instance

In [ ]:
from chi import context

context.version = "1.0"
context.choose_site(default="CHI@TACC")
context.choose_project()

In [ ]:
from chi import hardware

node_type = "compute_cascadelake_r"
available_nodes = hardware.get_nodes(node_type=node_type, filter_reserved=True)
if available_nodes:
    print(f"There currently are {len(available_nodes)} {node_type} nodes ready to use")
    hardware.show_nodes(available_nodes)
else:
    print(f"All {node_type} nodes are in use! You could use next_free_timeslot to see how long you need to wait, or use the calendar.")

In [ ]:
from chi import lease
from datetime import timedelta
import os

my_lease = lease.Lease(f"{os.getenv('USER')}-maicbf", duration=timedelta(hours=3))
my_lease.add_node_reservation(nodes=[available_nodes[0]])
my_lease.add_fip_reservation(1)
my_lease.submit(idempotent=True)

In [ ]:
from chi import server

my_server = server.Server(
    f"{os.getenv('USER')}-maicbf",
    reservation_id=my_lease.node_reservations[0]["id"],
    image_name="CC-Ubuntu22.04", # or use image_name
)
my_server.submit(idempotent=True)

In [ ]:
fip = my_lease.get_reserved_floating_ips()[0]
my_server.associate_floating_ip(fip)
my_server.check_connectivity(host=fip)

# Setup Environment

Create a Conda virtual environment to run this. Make sure Conda (or Miniconda) was installed beforehand. This is required to make sure the code can run smoothly using python 3.6

In [ ]:
#check if conda was installed
my_server.execute("ls /opt/conda/bin/conda || ls ~/miniconda3/bin/conda")

In [ ]:
#create virtual env
conda = "~/miniconda3/bin/conda"
my_server.execute(f"{conda} create -n maicbf python=3.6 -y") 
my_server.execute("ls ~/miniconda3/envs")

In [ ]:
#check if maicbf env ready to use
#to use the specific environment through this notebook, the path is required
cenv = "~/miniconda3/envs/maicbf/bin/"
my_server.execute(f"{cenv}python --version")

In [ ]:
#clone project repo
my_server.execute("git clone https://github.com/anisamsrh/MA-ICBF_Reproduction_Artifacts")

In [ ]:
#install project's required libraries
my_server.execute(f"cd maicbf && {cenv}pip install -r requirements.txt")

In [ ]:
#install library to store csv file in ObjectStorage ChameleonCloud
bucket_name = "maicbf"
my_server.execute(f"{cenv}pip install python-swiftclient python-keystoneclient")
my_server.execute(f"source ~/openrc.sh && {cenv}swift post {bucket_name}")

# Reproduction

Though i'm using .env to store my Telegram token which is used in all of the bash script files, it is not required to have one for this experiment to successfully run, so you can ignore the warning about missing .env file when you run the bash scripts. If you want to use Telegram notification, you can create a .env file in your home directory and add your token there which should contain: BOT_ID and USER_ID which you can get by following tutorial on the internet.

## Training

In [ ]:
#create directory for logging
my_server.execute("cd maicbf && mkdir -p train_logs && mkdir -p csv_data/losses && mkdir -p csv_data/trajectory")
#start tmux session
my_server.execute("tmux new -d -s train_batch")

In [ ]:
my_server.execute(f"tmux send-key -t train_batch 'conda activate maicbf && cd ~/maicbf' C-m")

In [ ]:
num_agents = 4
my_server.execute(f"tmux send-key -t train_batch 'python train.py --num_agents {num_agents}' C-m")

In [ ]:
#kill session when done
my_server.execute("tmux kill-session -t train_batch")

## Evaluation

### MA-ICBF

In [ ]:
my_server.execute(f"tmux new -d -s batch_eval")

In [ ]:
my_server.execute(f"tmux send-key -t batch_eval 'conda activate maicbf && cd ~/maicbf' C-m")

In [ ]:
my_server.execute(f"tmux send-key -t batch_eval 'bash ~/maicbf/batch_eval_all.sh' C-m")

In [ ]:
#this command is to watch current run_all.sh progress
result = my_server.execute("tmux capture-pane -pt batch_eval")
print(result)

In [ ]:
#kill session when done
my_server.execute("tmux kill-session -t batch_eval")

### MACBF (as a baseline)

In [ ]:
my_server.execute(f"tmux new -d -s batch_eval_baseline")

In [ ]:
my_server.execute(f"tmux send-key -t batch_eval_baseline 'conda activate maicbf && cd ~/maicbf/baselines/macbf/drones' C-m")

In [ ]:
my_server.execute(f"tmux send-key -t batch_eval_baseline 'bash ~/maicbf/baselines/macbf/drones/batch_eval.sh' C-m")

In [ ]:
#this command is to watch current run_all.sh progress
result = my_server.execute("tmux capture-pane -pt batch_eval_baseline")
print(result)

In [ ]:
#kill session when done
my_server.execute("tmux kill-session -t batch_eval_baseline")

## Result Visualization

In [ ]:
#run this command to upload csv files to ObjectStorage ChameleonCloud, 
# and you can download them from there for analysis

my_server.execute("bash ~/maicbf/upload_repo_data.sh")

In [ ]:
#download files

from chi import storage
import os

if not os.path.exists("./repo_data"):
    os.makedirs("./repo_data")

b = storage.ObjectBucket(f"{bucket_name}")
for obj in b.list_objects():
    print(f"Downloading {obj.name}")
    obj.download(f"./repo_data/{obj.name}")

In [ ]:
# run this to visualize the results, make sure you have the csv files 
# in repo_data folder before running this part

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

agents = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
collsion_avoidance = []
number_of_deadlocks = []
constraint_violations = []
compute_time = []

for a in agents:
    csv = pd.read_csv(f'./repo_data/Maze_{a}_agents.csv')
    collsion_avoidance.append(csv['collision_avoidance'].mean())
    number_of_deadlocks.append(csv['number_of_deadlocks'].sum())
    constraint_violations.append(csv['constraint_violations'].sum())
    compute_time.append(csv['compute_time'].mean())

def plot_collision_avoidance(agents, data, env='Empty'):
    plt.figure(figsize=(8, 5))
    plt.plot(agents, data, '-o', color='red', label='MAICBF')
    plt.xscale('log', base=2) 
    plt.xticks(agents, agents) 
    plt.grid(True, which="both", ls="--", alpha=0.5) 
    plt.xlabel('Number of Agents', fontweight='bold')
    plt.ylabel('Collision Avoidance (%)', fontweight='bold')
    plt.yticks(np.arange(20, 101, 20)) 
    plt.title(f'{env} environment')
    plt.ylim(20, 105)
    plt.xlim(7, 1100)
    plt.tight_layout() 
    plt.savefig(f'./repo_vis/{env}_collision_avoidance.png')

def plot_deadlocks(agents, data, env='Empty'):
    plt.figure(figsize=(8, 5))
    plt.plot(agents, data, '-o', color='red', label='MAICBF')
    plt.xscale('log', base=2) 
    plt.xticks(agents, agents) 
    plt.grid(True, which="both", ls="--", alpha=0.5) 
    plt.xlabel('Number of Agents', fontweight='bold')
    plt.ylabel('Avg. Number of Deadlocks', fontweight='bold')
    plt.yticks(np.arange(0, 201, 50)) 
    plt.title(f'{env} environment')
    plt.ylim(0, 210)
    plt.xlim(7, 1100)
    plt.tight_layout() 
    plt.savefig(f'./repo_vis/{env}_deadlocks.png')

def plot_constraint_violations(agents, data, env='Empty'):
    plt.figure(figsize=(8, 5))
    plt.plot(agents, data, '-o', color='red', label='MAICBF')
    plt.xscale('log', base=2) 
    plt.xticks(agents, agents) 
    plt.grid(True, which="both", ls="--", alpha=0.5) 
    plt.xlabel('Number of Agents', fontweight='bold')
    plt.ylabel('Input Constraint Violations', fontweight='bold')
    plt.yticks(np.arange(0, 501, 100))
    plt.title(f'{env} environment')
    plt.ylim(0, 530)
    plt.xlim(7, 1100)
    plt.tight_layout() 
    plt.savefig(f'./repo_vis/{env}_constraint_violations.png')

def plot_compute_time(agents, data, env='Empty'):
    plt.figure(figsize=(8, 5))
    plt.plot(agents, data, '-o', color='red', label='MAICBF')
    plt.xscale('log', base=2) 
    plt.xticks(agents, agents) 
    plt.grid(True, which="both", ls="--", alpha=0.5) 
    plt.xlabel('Number of Agents', fontweight='bold')
    plt.ylabel('Computational Time (s)', fontweight='bold')
    plt.yticks(np.arange(0, 41, 10)) 
    plt.title(f'{env} environment')
    plt.ylim(0, 43)
    plt.xlim(7, 1100)
    plt.tight_layout() 
    plt.savefig(f'./repo_vis/{env}_compute_time.png')

if not os.path.exists('./repo_vis'):
    os.makedirs('./repo_vis')
plot_collision_avoidance(agents, collsion_avoidance, env='Maze')
plot_deadlocks(agents, number_of_deadlocks, env='Maze')
plot_constraint_violations(agents, constraint_violations, env='Maze')
plot_compute_time(agents, compute_time, env='Maze')

collision_avoidance.clear()
number_of_deadlocks.clear()
constraint_violations.clear()
compute_time.clear()
for a in agents:
    csv = pd.read_csv(f'./repo_data/Empty_{a}_agents.csv')
    collsion_avoidance.append(csv['collision_avoidance'].mean())
    number_of_deadlocks.append(csv['number_of_deadlocks'].sum())
    constraint_violations.append(csv['constraint_violations'].sum())
    compute_time.append(csv['compute_time'].mean())

plot_collision_avoidance(agents, collsion_avoidance, env='Empty')
plot_deadlocks(agents, number_of_deadlocks, env='Empty')   
plot_constraint_violations(agents, constraint_violations, env='Empty')
plot_compute_time(agents, compute_time, env='Empty')